In [1]:
from pathlib import Path

import numpy as np
import optuna
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

from CustomSpeachDataset import CustomSpeachDataset
from SpeachClassifierModel import SpeachClassifierModel

/home/dom/python_global_venvs/ML/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
MODELS_DIR = Path("checkpoints")
MODELS_DIR.mkdir(exist_ok=True)

In [4]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset"), preload=True)
dataset.to(device)

Preloading dataset from disk...


In [5]:
train_val_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_val_indices]
train_val_dataset = torch.utils.data.Subset(dataset, train_val_indices)
train_val_sample_weights = all_sample_weights[train_val_indices]
len(train_val_dataset), len(y_train)

(14679, 14679)

In [6]:
def train_new_model(writer: SummaryWriter, fold: int, params: dict, epochs: int, train_loader: DataLoader, val_loader: DataLoader, save_dir: Path|None) -> SpeachClassifierModel:
    model = SpeachClassifierModel(params["dropout_rate"])
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    loss_fn = nn.CrossEntropyLoss()

    if save_dir:
        save_path = save_dir / f"fold{fold}.pth"

    best_val_loss = 99999999999.9

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            optimizer.zero_grad()

            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                y_pred = model(data)
                loss = loss_fn(y_pred, target)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            if save_dir:
                torch.save(model.state_dict(), save_path)

        model.best_val_loss = best_val_loss

        writer.add_scalar(f"Loss/train/fold{fold}", train_loss, epoch)
        writer.add_scalar(f"Loss/val/fold{fold}", val_loss, epoch)

    return model

In [7]:
def train_models_5cv(writer: SummaryWriter, params: dict, skf: StratifiedKFold, epochs: int, save_dir: Path|None=None) -> float:
    total_val_loss = 0.0

    for fold, (train_index, val_index) in enumerate(skf.split(y_train, y_train)):
        train_sampler = WeightedRandomSampler(
            weights=train_val_sample_weights[train_index],
            num_samples=len(train_val_sample_weights[train_index]),
            replacement=True,
        )
        train_subset = torch.utils.data.Subset(train_val_dataset, train_index)
        train_loader = DataLoader(train_subset, batch_size=32, sampler=train_sampler)

        val_subset = torch.utils.data.Subset(train_val_dataset, val_index)
        val_loader = DataLoader(val_subset, batch_size=1024)

        model = train_new_model(writer, fold, params, epochs, train_loader, val_loader, save_dir)
        total_val_loss += model.best_val_loss

    avl_val_loss = total_val_loss / skf.n_splits
    return avl_val_loss


In [8]:
def objective_train(trial: optuna.trial.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)

    current_params = trial.params

    # Fewer epochs for hyperparameters optimization process
    epochs = 20
    skf = StratifiedKFold(n_splits=5, shuffle=True)

    with SummaryWriter(f"runs/trial_{trial.number}/folds") as writer:
        avg_val_loss = train_models_5cv(writer, current_params, skf, epochs)

    metrics = {
        "hparams/avg_val_loss": avg_val_loss
    }

    with SummaryWriter(f"runs/optimization") as writer:
        writer.add_hparams(current_params, metrics, run_name=f"trial_{trial.number}")
        writer.add_scalar("Loss/avg_val_loss", avg_val_loss, trial.number)

    return avg_val_loss

In [9]:
study = optuna.create_study(study_name="Study", direction='minimize')

[I 2026-05-15 19:25:25,163] A new study created in memory with name: Study


In [ ]:
study.optimize(objective_train, n_trials=10)